<a href="https://colab.research.google.com/github/KattaLasya/HPCLab/blob/main/HPC_LAB06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install NUMA tools (Linux only)

In [ ]:
# Install numactl (required for NUMA binding)
!apt-get update
!apt-get install -y numactl

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.0 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,734 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,756 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:14 h

Check NUMA configuration of the system

In [ ]:
# Display NUMA hardware layout
!numactl --hardware

available: 1 nodes (0)
node 0 cpus: 0 1
node 0 size: 12975 MB
node 0 free: 8657 MB
node distances:
node   0 
  0:  10 


Create Multi-threaded Memory Access Program

In [ ]:
%%writefile numa_test.py

import threading
import numpy as np
import time
import os

# Configuration
ARRAY_SIZE = 200_000_000   # Large array (~1.6GB float64)
NUM_THREADS = os.cpu_count()

print(f"CPU Count: {NUM_THREADS}")
print(f"Allocating array of size: {ARRAY_SIZE}")

# Allocate shared memory
shared_array = np.zeros(ARRAY_SIZE, dtype=np.float64)

def worker(thread_id):
    chunk = ARRAY_SIZE // NUM_THREADS
    start = thread_id * chunk
    end = (thread_id + 1) * chunk if thread_id != NUM_THREADS - 1 else ARRAY_SIZE

    for i in range(start, end):
        shared_array[i] += 1

threads = []
start_time = time.time()

for i in range(NUM_THREADS):
    t = threading.Thread(target=worker, args=(i,))
    threads.append(t)
    t.start()

for t in threads:
    t.join()

end_time = time.time()
print(f"Execution Time: {end_time - start_time:.2f} seconds")

Writing numa_test.py


Run with Default Memory Allocation

In [ ]:
!python3 numa_test.py

CPU Count: 2
Allocating array of size: 200000000
Execution Time: 71.52 seconds


Run with Explicit NUMA Binding (CPU + Memory Node 0)

In [ ]:
!numactl --cpunodebind=0 --membind=0 python3 numa_test.py

CPU Count: 2
Allocating array of size: 200000000
Execution Time: 67.55 seconds


Run with Different CPU/Memory Node

In [ ]:
!numactl --cpunodebind=1 --membind=1 python3 numa_test.py